# Tutorial 5: Split Large Traces into Fragments

## Objective
Identify and split oversized chromatin traces into smaller fragments using K-means clustering.

## Scientific Context
Large traces can represent:
- **Multiple molecules**: Several chromatin fibers detected as one trace
- **Extended conformations**: Single molecule in very open state (rare)
- **Optical artifacts**: Multiple regions connected incorrectly

The splitting strategy:
- Compute **radius of gyration (Rg)** for each trace (measure of size)
- Identify traces with Rg > threshold (mean + N*std)
- Apply **K-means clustering** to split large traces into fragments
- Assign new Trace_IDs to each fragment

## Scripts Used
- `trace_splitter` - K-means based trace splitting

## Step 1: Setup

In [ ]:
import os

# Set up data paths
data_path = "/home/xdevos/Repositories/pyHi-M/traceratops/data/RUT"
clean_path = f"{data_path}/cleaned_output"
split_path = f"{data_path}/split_output"

os.makedirs(split_path, exist_ok=True)

# Input file from Tutorial 4
input_trace = f"{clean_path}/merged_traces_filtered_cleaned.ecsv"

print(f"Input: {input_trace}")
!ls -lh {input_trace}

## Step 2: Apply Trace Splitting

The `trace_splitter` command identifies large traces and splits them using K-means:
- **std_threshold** = 1.0: Splits traces with Rg > (mean + 1.0*std)
- **num_clusters** = 2: Each large trace split into 2 fragments

In [ ]:
!cd {split_path} && cp {input_trace} . && trace_splitter --input merged_traces_filtered_cleaned.ecsv --std_threshold 1.0 --num_clusters 2

## Step 3: Verify Output File

In [ ]:
!ls -lh {split_path}/*.ecsv | tail -2

## Step 4: Run QC Analysis on Filtered Traces

In [ ]:
split_file = f"{split_path}/merged_traces_filtered_cleaned_split.ecsv"
!trace_analyzer --input {split_file} -F {split_path}

## Understanding Radius of Gyration (Rg)

### Definition
Rg measures the average distance of all points from the center of mass:

```
Rg = sqrt( mean( (position - center_of_mass)^2 ) )
```

### Interpretation

| Rg Value | Meaning | Action |
|----------|---------|--------|
| Small (~0.2-0.3) | Compact | Keep |
| Medium (~0.3-0.5) | Normal | Keep |
| Large (> mean+std) | Over-extended or multiple molecules | SPLIT |

### Example
If mean_Rg = 0.257, std_Rg = 0.089, std_threshold = 1.0:
- Threshold = 0.257 + 1.0 * 0.089 = 0.346
- Traces with Rg > 0.346 get split

## Advanced: Different Thresholds

### std_threshold control splitting aggressiveness
- `--std_threshold 0.5`: More aggressive (split ~30% of traces)
- `--std_threshold 1.0`: Standard (split ~16% of traces)
- `--std_threshold 2.0`: Conservative (split ~2% of traces)

### num_clusters controls fragments per split
- `--num_clusters 2`: Split into 2 molecules (standard)
- `--num_clusters 3`: Split into 3 molecules (for very large traces)

In [ ]:
# Try different thresholds (uncomment to run)
# !cd {split_path} && trace_splitter --input merged_traces_filtered_cleaned.ecsv --std_threshold 0.5 --num_clusters 2
# !cd {split_path} && trace_splitter --input merged_traces_filtered_cleaned.ecsv --std_threshold 2.0 --num_clusters 2

print("Edit lines above to test different parameters")

## Summary

This tutorial demonstrated:

1. ✓ Computing radius of gyration to identify large traces
2. ✓ Applying statistical thresholding (mean + N*std)
3. ✓ K-means clustering to split oversized traces
4. ✓ Assigning new Trace_IDs to fragments

**Key Concepts:**
- Radius of gyration identifies unusually large traces
- K-means partitions large traces into biological fragments
- Each fragment gets a new Trace_ID
- std_threshold controls splitting aggressiveness

**Output:** Trace file with large traces split (`merged_traces_filtered_cleaned_split.ecsv`)

**Next Tutorial:** Assign masks and split by labels (tutorial_06_assign_masks_split_labels.ipynb)